# Packages import

In [22]:
import os
import re
import yaml
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

# Apollo scraper

In [23]:
url = "https://planzajec.uek.krakow.pl/index.php?typ=G&id=252671&okres=2"

In [24]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['username']
password = config['credentials']['password']
auth = HTTPBasicAuth(username, password)

In [25]:
group_id = input("Enter group ID: ")
url = "https://planzajec.uek.krakow.pl/index.php?typ=G&id=252671&okres=2"
response = requests.get(url, auth=auth)
response.encoding = "UTF-8"
print(response.status_code)

200


In [26]:
page_dom = BeautifulSoup(response.text, "html.parser")

In [27]:
group = page_dom.select_one("div.grupa").get_text()
print(group)

ZICSS1-1212


In [28]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(str(classes_tag))
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [29]:
classes.loc[classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"])]

,Termin,"Dzień, godzina",Przedmiot,Typ,Nauczyciel,Sala
1,2026-02-23,Pn 15:45 - 18:15 (3g.),Computer Programming 2,ćwiczenia,dr Katarzyna Wójcik,"Paw.A 013 lab. Win10, Office21"
4,2026-02-24,Wt 09:45 - 11:15 (2g.),Probability and Statistics,wykład,prof. dr hab. Andrzej Sokołowski,Paw.F 008
6,2026-02-24,Wt 15:00 - 16:30 (2g.),Discrete mathematics,wykład,dr Grzegorz Kosiorowski,Rakowicka 16 sala 11
7,2026-02-24,Wt 18:30 - 20:00 (2g.),Business Law,wykład,dr Jacek Lachner,Rakowicka 16 sala 11
9,2026-02-26,Cz 08:00 - 09:30 (2g.),Operating Systems and Computer Networks,wykład,prof. UEK dr hab. Joanna Wyrobek,Paw.F 008
...,...,...,...,...,...,...
292,2026-06-11,Cz 13:15 - 14:45 (2g.),Operating Systems and Computer Networks,ćwiczenia,prof. UEK dr hab. Joanna Wyrobek,"Paw.A 301a lab.Win10, Office21"
294,2026-06-11,Cz 16:45 - 18:15 (2g.),Operating Systems and Computer Networks,ćwiczenia,prof. UEK dr hab. Joanna Wyrobek,"Paw.A 07 lab. Win7, Office21"
296,2026-06-11,Cz 18:30 - 19:15 (1g.),Information Systems,ćwiczenia,mgr inż. Justyna Olczak,"Paw.A 013 lab. Win10, Office21"
297,2026-06-17,Śr 10:30 - 13:00 (3g.),Discrete mathematics,egzamin,dr Grzegorz Kosiorowski,Paw.C Nowa Aula


In [30]:
classes[['Day', 'Start time', 'Hyphen', 'End time', 'Duration']] = classes['Dzień, godzina'].str.split(' ',expand=True)

In [31]:
classes["Duration"] = classes["Duration"].map(lambda x: x.split("(")[1].split("g"[0]))

In [32]:
classes = classes.drop(["Hyphen"], axis=1)

In [33]:
classes["Sala"] = classes['Sala'].str.replace(
    r"(lab\.).*",
    r"\1",
    regex=True
)

In [34]:
if not os.path.exists("./schedules"):
    os.mkdir("./schedules")

In [35]:
group = "ZICSS1-1212"
classes.to_csv(f"schedules/{group}.csv", encoding="UTF-8")